# Phase 8 — Panel Construction, Treated Authors

Converts the publication histories into an author-year panel indexed on time
relative to retraction.

**Inputs:** `data/interim/phase04_papers.csv`,
`data/interim/phase04_author_queue.csv`,
`data/interim/phase02_author_paper.csv`, `data/interim/phase01_classified.csv`,
`data/raw/world_bank_income.csv`

**Outputs**

| File | Contents |
|---|---|
| `data/interim/phase08_authors.csv` | one row per author, with all covariates |
| `data/interim/phase08_panel_citations.csv` | author-year, citation window |
| `data/interim/phase08_panel_extended.csv` | author-year, full window |

## Two panels

| Panel | Extracted | Analysis window | Outcome |
|---|---|---|---|
| Citations | −7 to +3 | −3 to +3 | citations to other work |
| Extended | −7 to +6 | −6 to +6 | publications and exit |

The citation panel is shorter because OpenAlex citation counts begin in 2012,
so a six-year pre-period would exclude the earliest cohorts entirely.

## Career stage

Career length is measured from the earliest publication year in the extracted
histories. The `counts_by_year` field used for screening covers a rolling
window and understates career length; here the full history is available, so
the first year is observed rather than bounded.

In [1]:
import os

import numpy as np
import pandas as pd

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

PAPERS = "data/interim/phase04_papers.csv"
QUEUE = "data/interim/phase04_author_queue.csv"
AUTHOR_PAPER = "data/interim/phase02_author_paper.csv"
CLASSIFIED = "data/interim/phase01_classified.csv"
WORLD_BANK = "data/raw/world_bank_income.csv"

OUT_AUTHORS = "data/interim/phase08_authors.csv"
OUT_PANEL_CIT = "data/interim/phase08_panel_citations.csv"
OUT_PANEL_EXT = "data/interim/phase08_panel_extended.csv"

CIT_PRE, CIT_POST = 7, 3
CIT_ANALYSIS_PRE, CIT_ANALYSIS_POST = 3, 3
CIT_MIN_YEAR, CIT_MAX_YEAR = 2012, 2023

EXT_PRE, EXT_POST = 7, 6
EXT_ANALYSIS_PRE, EXT_ANALYSIS_POST = 6, 6
EXT_MIN_YEAR, EXT_MAX_YEAR = 2000, 2025


PRE_WINDOW = 7

REPEAT_AUTHOR_RULE = "single_only"

MIN_SUBJECT_AUTHORS = 300
MIN_CELL = 25
MAX_WORKS_PER_AUTHOR = 3000

ARMS = ["AUTHOR_MISCONDUCT", "HONEST_ERROR", "EDITORIAL_COMPROMISE"]

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 50)

print(f"citation panel  extract -{CIT_PRE}..+{CIT_POST}, "
      f"analyse -{CIT_ANALYSIS_PRE}..+{CIT_ANALYSIS_POST}, "
      f"years {CIT_MIN_YEAR}-{CIT_MAX_YEAR}")
print(f"extended panel  extract -{EXT_PRE}..+{EXT_POST}, "
      f"analyse -{EXT_ANALYSIS_PRE}..+{EXT_ANALYSIS_POST}, "
      f"years {EXT_MIN_YEAR}-{EXT_MAX_YEAR}")
print(f"repeat authors  {REPEAT_AUTHOR_RULE}")
print(f"pre-window      {PRE_WINDOW} years")

citation panel  extract -7..+3, analyse -3..+3, years 2012-2023
extended panel  extract -7..+6, analyse -6..+6, years 2000-2025
repeat authors  single_only
pre-window      7 years


## Load

In [2]:
papers = pd.read_csv(PAPERS, low_memory=False)
n_raw = len(papers)
papers = papers.drop_duplicates(subset=["author_id", "work_id"])
if len(papers) < n_raw:
    print(f"removed {n_raw - len(papers):,} duplicate author-work rows")
papers["author_id"] = papers.author_id.astype(str)
papers = papers.dropna(subset=["pub_year"])
papers["pub_year"] = papers.pub_year.astype(int)

queue = pd.read_csv(QUEUE, low_memory=False)
queue["author_id"] = queue.author_id.astype(str)

ap = pd.read_csv(AUTHOR_PAPER, low_memory=False)
ap["author_id"] = ap.author_id.astype(str)

if MAX_WORKS_PER_AUTHOR is not None:
    per = papers.groupby("author_id").size()
    over = per[per > MAX_WORKS_PER_AUTHOR]
    if len(over):
        print(f"excluded {len(over):,} authors above "
              f"{MAX_WORKS_PER_AUTHOR:,} works "
              f"({len(over) / papers.author_id.nunique():.3%}), "
              f"removing {int(over.sum()):,} rows")
        print(f"  largest: {over.max():,} works")
        papers = papers[~papers.author_id.isin(over.index)]
        queue = queue[~queue.author_id.isin(over.index)]

print(f"paper rows        {len(papers):,}")
print(f"authors extracted {papers.author_id.nunique():,}")
print(f"queue             {len(queue):,}")
print(f"author-paper rows {len(ap):,}")

removed 31,257 duplicate author-work rows
excluded 32 authors above 3,000 works (0.057%), removing 130,219 rows
  largest: 9,462 works
paper rows        7,643,591
authors extracted 55,789
queue             55,789
author-paper rows 78,889


## Covariates

The attribution arm is assigned in Phase 1 and carried through Phase 3. It is
not recomputed here, so the taxonomy has a single definition.

In [3]:
def career_band(years):
    return pd.cut(years, bins=[0, 5, 10, 20, np.inf],
                  labels=["early (<5y)", "mid (5-10y)",
                          "senior (10-20y)", "veteran (20y+)"],
                  right=False)


def lag_band(years):
    return pd.cut(years, bins=[-np.inf, 1, 3, np.inf],
                  labels=["fast (<1y)", "medium (1-3y)", "slow (3y+)"],
                  right=False)


auth = queue.rename(columns={"first_category": "arm"}).copy()

first_pub = papers.groupby("author_id").pub_year.min()
auth["first_pub_year_extract"] = auth.author_id.map(first_pub)
auth["career_years_at_retraction"] = (auth.first_retraction_year
                                      - auth.first_pub_year_extract)
auth = auth[auth.career_years_at_retraction >= 0]
auth["career_band"] = career_band(auth.career_years_at_retraction)
print(f"career stage computable  {auth.career_band.notna().mean():.1%}")

ry = auth.set_index("author_id").first_retraction_year
p = papers.copy()
p["retraction_year"] = p.author_id.map(ry)
pre = p[(p.pub_year >= p.retraction_year - PRE_WINDOW) &
        (p.pub_year < p.retraction_year)]
auth["pre_publications"] = (auth.author_id.map(pre.groupby("author_id").size())
                              .fillna(0).astype(int))

career stage computable  100.0%


In [4]:
ccol = next((c for c in ["country", "last_country"] if c in auth.columns), None)
print(f"country present          {auth[ccol].notna().mean():.1%}")

if os.path.isfile(WORLD_BANK):
    wb = pd.read_csv(WORLD_BANK)
    iso = next((c for c in wb.columns if "iso" in c.lower() or c == "Code"), None)
    grp = next((c for c in wb.columns
                if "income" in c.lower() or "group" in c.lower()), None)
    yr = next((c for c in wb.columns if "year" in c.lower()), None)
    if iso and grp:
        if yr:
            wb = wb.sort_values(yr)
        inc = dict(zip(wb[iso].astype(str).str.upper(), wb[grp].astype(str)))
        auth["income"] = auth[ccol].astype(str).str.upper().map(inc)
        # Four tiers leave too few low-income authors to estimate on.
        auth["income_group"] = auth.income.map(
            {"H": "higher", "UM": "higher", "LM": "lower", "L": "lower"})
        print(f"income tier present      {auth.income.notna().mean():.1%}")
        print(auth.income.value_counts().to_string())
    else:
        auth["income"] = auth["income_group"] = None
else:
    print(f"  [!] {WORLD_BANK} not found; income_group unavailable")
    auth["income"] = auth["income_group"] = None

country present          91.2%


In [5]:
# Subject domain, from the retracted paper. A paper spanning several domains
# takes the first listed.
if os.path.isfile(CLASSIFIED):
    cls = pd.read_csv(CLASSIFIED, low_memory=False)
    if "Subject" in cls.columns and "Journal" in cls.columns:
        dom = cls.Subject.astype(str).str.extract(r"\(([^)]+)\)")[0]
        multi = cls.Subject.astype(str).str.count(r"\(").gt(1).mean()
        print(f"papers spanning >1 domain {multi:.1%}, first listed used")
        by_journal = dict(zip(cls.Journal.astype(str), dom))
        jcol = next((c for c in ["rw_journal", "Journal"]
                     if c in auth.columns), None)
        if jcol:
            auth["subject_domain"] = auth[jcol].astype(str).map(by_journal)
            print(f"subject domain present    "
                  f"{auth.subject_domain.notna().mean():.1%}")

if "subject_domain" in auth.columns:
    counts = auth.subject_domain.value_counts()
    small = counts[counts < MIN_SUBJECT_AUTHORS].index.tolist()
    auth["subject_group"] = auth.subject_domain.where(
        ~auth.subject_domain.isin(small), "OTHER")
    if small:
        print(f"merged into OTHER: {small}")
    print(f"subject groups: {auth.subject_group.value_counts().to_dict()}")
else:
    auth["subject_group"] = None

lag = (ap.dropna(subset=["pub_year", "retraction_year"])
         .assign(lag=lambda d: d.retraction_year - d.pub_year)
         .groupby("author_id").lag.min())
auth["lag_years"] = auth.author_id.map(lag)
auth["lag_band"] = lag_band(auth.lag_years)
print(f"detection lag computable  {auth.lag_years.notna().mean():.1%}")

print(f"\npre-retraction publications over {PRE_WINDOW} years")
print(auth.pre_publications.describe().round(1).to_string())
print(f"\ncareer band")
print(auth.career_band.value_counts().to_string())

papers spanning >1 domain 85.9%, first listed used
subject domain present    100.0%
merged into OTHER: ['HUM']
subject groups: {'BLS': 33027, 'B/T': 7548, 'HSC': 6839, 'PHY': 6248, 'ENV': 1424, 'SOC': 587, 'OTHER': 116}
detection lag computable  100.0%

pre-retraction publications over 7 years
count    55789.0
mean        42.9
std         68.7
min          0.0
25%          8.0
50%         21.0
75%         50.0
max       2672.0

career band
career_band
veteran (20y+)     25516
senior (10-20y)    18810
mid (5-10y)         8463
early (<5y)         3000


## Sample definition

An author with several retractions has a post-period containing more than one
event, so the estimate for them is not a single treatment effect. The primary
sample restricts to authors with exactly one. The cost is reported rather than
assumed.

In [6]:
if "n_retractions" not in auth.columns:
    print("  [!] n_retractions absent; every author treated as single")
    auth["n_retractions"] = 1

if REPEAT_AUTHOR_RULE == "single_only":
    auth["in_primary"] = auth.n_retractions == 1
elif REPEAT_AUTHOR_RULE == "first_only":
    auth["in_primary"] = True
else:
    raise SystemExit(f"unknown REPEAT_AUTHOR_RULE: {REPEAT_AUTHOR_RULE}")

print(f"authors                {len(auth):,}")
print(f"  one retraction       {int((auth.n_retractions == 1).sum()):,}")
print(f"  more than one        {int((auth.n_retractions > 1).sum()):,}")
print(f"  in the primary sample {int(auth.in_primary.sum()):,}  "
      f"({auth.in_primary.mean():.1%})")

print(f"\nby arm, primary sample")
print(auth[auth.in_primary].arm.value_counts().to_string())

authors                55,789
  one retraction       48,782
  more than one        7,007
  in the primary sample 48,782  (87.4%)

by arm, primary sample
arm
AUTHOR_MISCONDUCT       25361
HONEST_ERROR            10923
EDITORIAL_COMPROMISE     7128
UNCONFIRMED_CONCERNS     4302
ETHICS_VIOLATION          723
UNCLASSIFIED              345


## Panel assembly

One row per author per year across the extraction window, with zeros where an
author published nothing.

`balanced` marks authors present at every event time in the analysis window.
Without it a coefficient could reflect authors entering or leaving the sample
rather than a treatment effect.

In [7]:
def parse_counts(raw):
    """counts_by_year is stored compactly as 'YYYY:n|YYYY:n'."""
    if not isinstance(raw, str) or not raw.strip():
        return {}
    out = {}
    for part in raw.split("|"):
        if ":" not in part:
            continue
        y, _, c = part.partition(":")
        try:
            out[int(y)] = int(float(c))
        except ValueError:
            continue
    return out


def build_panel(papers, auth, pre, post, min_year, max_year,
                a_pre, a_post, with_citations, label):
    print(f"\n{label}")

    ry = auth.set_index("author_id").first_retraction_year.to_dict()

    pubs = (papers.groupby(["author_id", "pub_year"]).size()
                  .reset_index(name="publications"))
    pub_lookup = {a: g.set_index("pub_year").publications
                  for a, g in pubs.groupby("author_id")}

    cit_lookup = {}
    if with_citations:
        for aid, grp in papers.groupby("author_id"):
            tally = {}
            for raw in grp.counts_by_year:
                for y, c in parse_counts(raw).items():
                    tally[y] = tally.get(y, 0) + c
            cit_lookup[aid] = tally

    rows, dropped = [], 0
    for aid, R in ry.items():
        R = int(R)
        pl = pub_lookup.get(aid, pd.Series(dtype=int))
        cl = cit_lookup.get(aid, {})
        for y in range(R - pre, R + post + 1):
            if not (min_year <= y <= max_year):
                dropped += 1
                continue
            n = int(pl.get(y, 0))
            row = {"author_id": aid, "year": y, "event_time": y - R,
                   "publications": n, "active": int(n > 0)}
            if with_citations:
                row["citations"] = int(cl.get(y, 0))
            rows.append(row)

    panel = pd.DataFrame(rows)
    print(f"  panel rows {len(panel):,}  ({dropped:,} dropped outside "
          f"{min_year}-{max_year})")

    need = a_pre + a_post + 1
    in_window = panel[(panel.event_time >= -a_pre) & (panel.event_time <= a_post)]
    span = in_window.groupby("author_id").event_time.nunique()
    balanced = set(span[span == need].index)
    panel["balanced"] = panel.author_id.isin(balanced)
    print(f"  balanced on -{a_pre}..+{a_post}: {len(balanced):,} authors "
          f"({len(balanced) / max(auth.author_id.nunique(), 1):.1%})")

    cov = [c for c in ["arm", "first_position", "n_retractions", "in_primary",
                       "career_band", "career_years_at_retraction",
                       "pre_publications", "income_group", "subject_group",
                       "lag_band", "lag_years", "first_retraction_year",
                       "rw_journal", "rw_publisher", "source_id"]
           if c in auth.columns]
    return panel.merge(auth[["author_id"] + cov], on="author_id", how="left")


panel_cit = build_panel(papers, auth, CIT_PRE, CIT_POST,
                        CIT_MIN_YEAR, CIT_MAX_YEAR,
                        CIT_ANALYSIS_PRE, CIT_ANALYSIS_POST,
                        with_citations=True, label="CITATION PANEL")

panel_ext = build_panel(papers, auth, EXT_PRE, EXT_POST,
                        EXT_MIN_YEAR, EXT_MAX_YEAR,
                        EXT_ANALYSIS_PRE, EXT_ANALYSIS_POST,
                        with_citations=False, label="EXTENDED PANEL")


CITATION PANEL
  panel rows 531,571  (82,108 dropped outside 2012-2023)
  balanced on -3..+3: 29,871 authors (53.5%)

EXTENDED PANEL
  panel rows 706,946  (74,100 dropped outside 2000-2025)
  balanced on -6..+6: 22,535 authors (40.4%)


## The estimation sample

In [8]:
def summarise(panel, label):
    print(f"\n{label}")
    bal = panel[panel.balanced & panel.in_primary]
    print(f"balanced and in the primary sample: "
          f"{bal.author_id.nunique():,} authors, {len(bal):,} rows\n")

    for col, name in [("arm", "arm"), ("first_position", "byline position"),
                      ("career_band", "career stage"),
                      ("income_group", "income group"),
                      ("subject_group", "subject group"),
                      ("lag_band", "detection lag")]:
        if col not in bal.columns or bal[col].isna().all():
            continue
        vc = bal.groupby("author_id")[col].first().value_counts()
        print(f"by {name}")
        print("  " + vc.to_string().replace("\n", "\n  "))
        print()

    print("by retraction year")
    print("  " + bal.groupby("author_id").first_retraction_year.first()
            .value_counts().sort_index().to_string().replace("\n", "\n  "))


summarise(panel_ext, "EXTENDED PANEL, publications and exit")
summarise(panel_cit, "CITATION PANEL")


EXTENDED PANEL, publications and exit
balanced and in the primary sample: 19,265 authors, 269,710 rows

by arm
  arm
  AUTHOR_MISCONDUCT       9617
  HONEST_ERROR            4894
  EDITORIAL_COMPROMISE    2783
  UNCONFIRMED_CONCERNS    1457
  ETHICS_VIOLATION         327
  UNCLASSIFIED             187

by byline position
  first_position
  middle    12190
  first      3770
  last       3305

by career stage
  career_band
  veteran (20y+)     8791
  senior (10-20y)    6166
  mid (5-10y)        3124
  early (<5y)        1184

by subject group
  subject_group
  BLS      11608
  HSC       2996
  PHY       2446
  B/T       1508
  ENV        449
  SOC        212
  OTHER       46

by detection lag
  lag_band
  medium (1-3y)    7926
  slow (3y+)       7119
  fast (<1y)       4220

by retraction year
  first_retraction_year
  2015.0    3345
  2016.0    3418
  2017.0    3542
  2018.0    3860
  2019.0    5100

CITATION PANEL
balanced and in the primary sample: 25,486 authors, 245,768 rows

by ar

## Cell sizes for the pre-specified splits

A heterogeneity split below the minimum cell size will not estimate reliably.
Reported so those comparisons are dropped rather than published with intervals
wide enough to accommodate anything.

In [9]:
bal = panel_ext[panel_ext.balanced & panel_ext.in_primary]
per_author = bal.groupby("author_id").first()

for col, name in [("career_band", "career stage"),
                  ("income_group", "income group"),
                  ("subject_group", "subject group"),
                  ("lag_band", "detection lag")]:
    if col not in per_author.columns or per_author[col].isna().all():
        continue
    tab = pd.crosstab(per_author[col], per_author.arm)
    keep = [a for a in ARMS if a in tab.columns]
    tab = tab[keep] if keep else tab
    print(f"\narm x {name}")
    print(tab.to_string())
    thin = int((tab < MIN_CELL).sum().sum())
    if thin:
        print(f"  [!] {thin} cell(s) below {MIN_CELL} authors; those splits "
              f"will not estimate reliably")


arm x career stage
arm              AUTHOR_MISCONDUCT  HONEST_ERROR  EDITORIAL_COMPROMISE
career_band                                                           
early (<5y)                    586           286                   202
mid (5-10y)                   1640           708                   462
senior (10-20y)               3069          1552                   964
veteran (20y+)                4322          2348                  1155

arm x subject group
arm            AUTHOR_MISCONDUCT  HONEST_ERROR  EDITORIAL_COMPROMISE
subject_group                                                       
B/T                          813           196                   378
BLS                         5939          3076                  1416
ENV                          264            74                    78
HSC                         1275           806                   536
OTHER                         22             6                    18
PHY                         1214           679    

## Write

In [10]:
auth.to_csv(OUT_AUTHORS, index=False)
print(f"{OUT_AUTHORS}: {len(auth):,} authors")

panel_cit.to_csv(OUT_PANEL_CIT, index=False)
print(f"{OUT_PANEL_CIT}: {len(panel_cit):,} rows")

panel_ext.to_csv(OUT_PANEL_EXT, index=False)
print(f"{OUT_PANEL_EXT}: {len(panel_ext):,} rows")

bal = panel_ext[panel_ext.balanced & panel_ext.in_primary]
in_arms = bal[bal.arm.isin(ARMS)]
print(f"\nestimation sample, three arms: "
      f"{in_arms.author_id.nunique():,} authors, {len(in_arms):,} rows")

data/interim/phase08_authors.csv: 55,789 authors
data/interim/phase08_panel_citations.csv: 531,571 rows
data/interim/phase08_panel_extended.csv: 706,946 rows

estimation sample, three arms: 17,294 authors, 242,116 rows
